# 5장 실습 ② — 가중치 초기화

**Keras 3 판**

*"기준을 아무 데나 하나 잡습니다"* — 2장 §2.2의 그 "아무 데나"를 봅니다.
본문 §5.3의 표를 이 노트북이 만듭니다.

## 5.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 5.1 실험대 — 소용돌이

이 문제의 **바닥**을 먼저 재 둡니다. 직선 하나로 최대 얼마나 맞히는가.
앞으로 나오는 숫자가 이 값 근처면 **"사실상 직선 하나"**라는 뜻입니다.

In [ ]:
# 소용돌이 — 하이퍼파라미터가 결과를 실제로 바꾸는 문제.
# 사과 데이터는 무엇을 해도 0.95가 나와서 이 장의 실험대가 될 수 없다.
x, y = data.spirals(n=1600, seed=42)
s = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)
print(s.summary())

plot.scatter2d(s.x_train, s.y_train, class_names=("무리 0", "무리 1"),
               xlabel="$x_1$", ylabel="$x_2$", title="소용돌이 — 이 장의 실험대")
plt.show()

# 이 문제의 바닥: 직선 하나로 최대 얼마나 맞히는가
rng = np.random.default_rng(0)
best = 0.0
for _ in range(4000):
    w, b = rng.normal(size=2), rng.normal() * 3
    a = metrics.accuracy(y, (x @ w + b > 0).astype(int))
    best = max(best, a, 1 - a)
dlbook.record("ch05_spiral_single_line_acc", best)
print("→ 0.7 근처의 결과가 나오면 '사실상 직선 하나'라는 뜻입니다.")

## 5.2 학습 함수 — 여기만 판마다 다릅니다

아래 셀 하나가 이 판의 방식으로 모델을 만들고 학습시킵니다.
**이 아래의 모든 셀은 세 판이 같습니다.**

In [ ]:
import keras
from keras import layers

dlbook.set_seed(42)

def train_model(depth=3, units=32, act="relu", init="glorot_uniform",
                lr=0.01, opt="adam", epochs=60, bs=32, seed=42):
    """모델을 만들어 학습시키고 (시험 정확도, history)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    model = keras.Sequential(
        [layers.Input(shape=(2,))]
        + [layers.Dense(units, activation=act, kernel_initializer=init)
           for _ in range(depth)]
        + [layers.Dense(1, activation="sigmoid", kernel_initializer=init)]
    )
    optimizer = {"adam": keras.optimizers.Adam,
                 "sgd": keras.optimizers.SGD,
                 "rmsprop": keras.optimizers.RMSprop}[opt](learning_rate=lr)
    model.compile(optimizer=optimizer, loss="binary_crossentropy",
                  metrics=["accuracy"])
    h = model.fit(s.x_train, s.y_train, validation_data=(s.x_val, s.y_val),
                  epochs=dlbook.smoke.epochs(epochs), batch_size=bs, verbose=0)
    pred = (model.predict(s.x_test, verbose=0).reshape(-1) > 0.5).astype("int64")
    return metrics.accuracy(s.y_test, pred), h.history

## 5.3 가중치 초기화

*"기준을 아무 데나 하나 잡습니다"* — 2장 §2.2의 그 "아무 데나"입니다.

**아무 데나가 정말 아무 데나가 아닙니다.**

In [ ]:
# 은닉층 5개, 시그모이드. 초기화 방법만 바꾼다.
for init, name in (("zeros", "전부 0으로"),
                   ("random_normal", "정규분포 0.05"),
                   ("glorot_uniform", "Xavier/Glorot"),
                   ("he_normal", "He")):
    acc, h = train_model(depth=5, act="sigmoid", init=init, lr=0.01)
    print(f"{name:<16} 시험 정확도 {acc:.3f}   최종 학습 손실 {h['loss'][-1]:.4f}")
    dlbook.record(f"ch05_init_{init}_acc", acc)

## 정리

- **가중치는 서로 달라야 합니다.** 전부 0이면 한 층의 노드가 전부 똑같이
  갱신되어, 32개를 만들어 놓고 1개를 쓰는 것과 같아집니다.
- **층 크기에 맞는 폭이어야 합니다.** 너무 작으면 신호가 사그라들고, 너무
  크면 폭발합니다. Xavier(시그모이드·tanh)와 He(ReLU)가 그 폭을 계산해 줍니다.

### 연습

1. `zeros` 로 초기화한 모델에서 한 층의 가중치를 출력해, **모든 노드가
   같은 값**임을 확인하십시오.
2. 편향만 0으로 두고 가중치는 난수로 두면 어떻게 됩니까. 왜 그렇습니까.
3. ReLU 모델에 Xavier를 주면 He를 줄 때와 얼마나 다릅니까.